# PLC Timestamp Probe - TX2

Notebook para probar desde una maquina dentro de la VPN si podemos leer tags del sistema de vision desde OPC UA.

Endpoint TX2 usado por default:

`opc.tcp://10.14.6.48:49320`

Primero corre las celdas en orden. Para AS20 > VisionSystem ya dejamos configurados `MeasureLength` y `VisionWD`.

Importante: `VisionWD` es el watchdog del PLC/sistema de vision; si cambia, la comunicacion esta viva. Segun Kepware tiene scan rate de 50 ms y el PLC lo actualiza alrededor de cada 69 ms.

Contexto: la app no habla directo con el PLC; habla con Kepware por OPC UA, y Kepware habla con ControlLogix. Para esta prueba usamos el grupo `Connectivity > ControlLogix > AS20 > VisionSystem`, no `TX2MH > Scanning`. El objetivo es validar vida con `VisionWD` y revisar si `MeasureLength` sirve como medicion o trigger para sincronizar con el video.

In [ ]:
%pip install asyncua

In [ ]:
# Configuracion
ENDPOINT = "opc.tcp://10.14.6.48:49320"  # TX2 OPC UA server / BRJTXQMOSTX2OPC
# Si DNS funciona en la VPN, tambien puede usarse:
# ENDPOINT = "opc.tcp://BRJTXQMOSTX2OPC.BARNSTXPROD.LOCAL:49320"

# Tags vistos en Kepware: Connectivity > ControlLogix > AS20 > VisionSystem
MEASURE_LENGTH_NODE = "ns=2;s=ControlLogix.AS20.VisionSystem.MeasureLength"
VISION_WD_NODE = "ns=2;s=ControlLogix.AS20.VisionSystem.VisionWD"

NODE_IDS = [
    MEASURE_LENGTH_NODE,
    VISION_WD_NODE,
]

# Palabras para buscar tags candidatos si todavia no sabemos el NodeId exacto.
KEYWORDS = ["measure", "length", "vision", "wd", "watchdog", "timestamp", "time", "date", "execution", "exec", "scan", "trigger"]

MAX_DEPTH = 7
MAX_NODES = 5000
TIMEOUT_SECONDS = 8.0

In [ ]:
import asyncio
import json
from collections import deque
from datetime import datetime, timezone
from typing import Any

from asyncua import Client, ua


def now_utc() -> str:
    return datetime.now(timezone.utc).isoformat(timespec="milliseconds")


def clean(value: Any) -> Any:
    if isinstance(value, datetime):
        return value.isoformat()
    if isinstance(value, bytes):
        return value.hex()
    if isinstance(value, (str, int, float, bool)) or value is None:
        return value
    if isinstance(value, (list, tuple)):
        return [clean(item) for item in value]
    return str(value)


def matches_keywords(record: dict[str, Any], keywords: list[str]) -> bool:
    haystack = " ".join(
        str(record.get(key, "")) for key in ("node_id", "browse_name", "display_name", "path")
    ).lower()
    return any(keyword.lower() in haystack for keyword in keywords)


async def read_node(client: Client, node_id: str) -> dict[str, Any]:
    node = client.get_node(node_id)
    data_value = await node.read_data_value()
    try:
        display_name = (await node.read_display_name()).Text
    except Exception:
        display_name = ""
    try:
        browse_name = (await node.read_browse_name()).Name
    except Exception:
        browse_name = ""
    return {
        "node_id": node_id,
        "browse_name": browse_name,
        "display_name": display_name,
        "status": str(data_value.StatusCode),
        "value": clean(data_value.Value.Value),
        "source_timestamp": clean(data_value.SourceTimestamp),
        "server_timestamp": clean(data_value.ServerTimestamp),
        "read_utc": now_utc(),
    }


async def browse_candidates(
    client: Client,
    keywords: list[str],
    max_depth: int = 7,
    max_nodes: int = 5000,
) -> list[dict[str, Any]]:
    objects = client.get_objects_node()
    queue = deque([(objects, "Objects", 0)])
    seen: set[str] = set()
    candidates: list[dict[str, Any]] = []
    visited = 0

    while queue and visited < max_nodes:
        node, path, depth = queue.popleft()
        node_key = node.nodeid.to_string()
        if node_key in seen:
            continue
        seen.add(node_key)
        visited += 1

        try:
            children = await node.get_children()
        except Exception:
            continue

        for child in children:
            child_id = child.nodeid.to_string()
            try:
                browse_name = (await child.read_browse_name()).Name
            except Exception:
                browse_name = ""
            try:
                display_name = (await child.read_display_name()).Text
            except Exception:
                display_name = ""
            try:
                node_class = await child.read_node_class()
            except Exception:
                node_class = None

            child_path = f"{path}/{display_name or browse_name or child_id}"
            record = {
                "node_id": child_id,
                "browse_name": browse_name,
                "display_name": display_name,
                "path": child_path,
                "node_class": str(node_class),
            }

            if node_class == ua.NodeClass.Variable and matches_keywords(record, keywords):
                try:
                    record.update(await read_node(client, child_id))
                except Exception as exc:
                    record["read_error"] = str(exc)
                    record["read_utc"] = now_utc()
                candidates.append(record)

            if depth + 1 <= max_depth and child_id not in seen:
                queue.append((child, child_path, depth + 1))

    candidates.sort(key=lambda item: item.get("path", ""))
    return candidates


def print_item(item: dict[str, Any]) -> None:
    print("-" * 90)
    print(f"path:   {item.get('path', '')}")
    print(f"node:   {item.get('node_id')}")
    print(f"name:   {item.get('display_name') or item.get('browse_name')}")
    print(f"value:  {item.get('value')!r}")
    print(f"source: {item.get('source_timestamp')}")
    print(f"server: {item.get('server_timestamp')}")
    if item.get("read_error"):
        print(f"error:  {item['read_error']}")

## 1. Buscar tags candidatos

Corre esta celda si quieres confirmar nombres o explorar mas tags. La salida importante es `node:`. Para lo que vimos en Kepware, deberian aparecer `ControlLogix.AS20.VisionSystem.MeasureLength` y `ControlLogix.AS20.VisionSystem.VisionWD`.

In [ ]:
async with Client(url=ENDPOINT, timeout=TIMEOUT_SECONDS) as client:
    candidates = await browse_candidates(
        client,
        keywords=KEYWORDS,
        max_depth=MAX_DEPTH,
        max_nodes=MAX_NODES,
    )

print(f"Candidates found: {len(candidates)}")
for item in candidates[:50]:
    print_item(item)

## 2. Leer MeasureLength y VisionWD

Corre esta celda para leer directo los tags de `AS20 > VisionSystem`. Lo que queremos validar es:

- `MeasureLength.value`: medicion que ya esta publicando el PLC/Kepware.
- `VisionWD.value`: contador watchdog; debe cambiar continuamente si el PLC/sistema de vision esta vivo.
- `source_timestamp`: hora del dato segun OPC UA.
- `server_timestamp`: hora en la que el servidor OPC UA publico/entrego el valor.

In [ ]:
if not NODE_IDS:
    print("NODE_IDS esta vacio. Pega aqui los NodeIds candidatos y vuelve a correr:")
    print('NODE_IDS = ["ns=2;s=..."]')
else:
    async with Client(url=ENDPOINT, timeout=TIMEOUT_SECONDS) as client:
        direct_reads = []
        for node_id in NODE_IDS:
            try:
                direct_reads.append(await read_node(client, node_id))
            except Exception as exc:
                direct_reads.append({"node_id": node_id, "error": str(exc), "read_utc": now_utc()})

    for item in direct_reads:
        print_item(item)

## 3. Monitorear cada segundo

Usa esta celda para observar `MeasureLength` y `VisionWD` con una frecuencia tranquila. Detenla con el boton stop del notebook.

In [ ]:
WATCH_SECONDS = 1.0

if not NODE_IDS:
    print("Primero configura NODE_IDS con el tag exacto o candidato.")
else:
    async with Client(url=ENDPOINT, timeout=TIMEOUT_SECONDS) as client:
        while True:
            print("=" * 90)
            print(f"Read UTC: {now_utc()}")
            for node_id in NODE_IDS:
                try:
                    print_item(await read_node(client, node_id))
                except Exception as exc:
                    print(f"{node_id}: ERROR {exc}")
            await asyncio.sleep(WATCH_SECONDS)

## 4. Monitoreo rapido de VisionWD

Esta celda intenta muestrear el watchdog cada 70 ms, cercano al ritmo que comentaste. En Windows/Jupyter puede haber jitter, pero sirve para validar que el valor esta cambiando rapido.

In [ ]:
FAST_WATCH_SECONDS = 0.07
SAMPLES = 100

async with Client(url=ENDPOINT, timeout=TIMEOUT_SECONDS) as client:
    last_value = None
    changes = 0
    for i in range(SAMPLES):
        item = await read_node(client, VISION_WD_NODE)
        value = item.get("value")
        if last_value is not None and value != last_value:
            changes += 1
        last_value = value
        print(f"{i:03d} | value={value!r} | source={item.get('source_timestamp')} | server={item.get('server_timestamp')}")
        await asyncio.sleep(FAST_WATCH_SECONDS)

print(f"Changes detected: {changes} / {SAMPLES - 1}")

## Nota importante

Para sincronizar con el video, `VisionWD` solo confirma que el PLC/sistema de vision esta vivo. Para el momento exacto de medicion, hay que usar el cambio/evento asociado a `MeasureLength` o agregar un tag de timestamp/event trigger en `AS20 > VisionSystem` si el PLC lo expone.